# Train kinematic-conditioned SD (LoRA) on Google Colab

Use this notebook to run `scripts/train_sd.py` on a **GPU** (CUDA), save checkpoints to **Google Drive**, and copy them back to your Mac / Cursor workspace.

**Before you run**
- **Runtime → Change runtime type → GPU** (T4 / L4 / A100).
- **Dataset:** Section 1 sets **`JIGSAWS_ZIP`** to your uploaded zip on Drive; **section 4c** unzips it next to that file and sets **`DATA_ROOT`**. You can instead set **`DATA_ROOT`** to an already-unzipped folder and skip 4c (see the config cell).
- The JIGSAWS root must contain the **suturing** task and **experimental_setup** trees (often **`Suturing/`** and **`Experimental_setup/`** on disk; resolved case-insensitively like `train_sd.py`).
- **Google Drive shortcuts** are unreliable in Colab; a **real zip or folder copy** on Drive is preferred.
- **Code source (pick one):** if **`REPO_URL`** is set, the repo is **cloned** into `CLONE_TARGET` and **`PROJECT_ROOT_DRIVE` is ignored**. If **`REPO_URL`** is empty, the notebook uses **`PROJECT_ROOT_DRIVE`** (a full copy of SS-SD you uploaded to Drive).
- Optional: [Hugging Face token](https://huggingface.co/settings/tokens) if downloads are rate-limited (`Settings → Secrets` in Colab, or set `HF_TOKEN` in a cell).

**After training**
- Checkpoints live under `SAVE_DIR` on Drive (`step_*.pt`, `args.json`). Download that folder in Drive’s web UI, or sync Drive to your machine, and place it under your repo’s `checkpoints/` (e.g. `checkpoints/suturing_expert_lora/`).

## 1. Configuration (edit this cell)

In [ ]:
# --- Paths: edit for your Drive layout ---
DRIVE = "/content/drive/MyDrive"

# JIGSAWS zip on Drive (upload path must match). Section 4c unzips beside this file and sets DATA_ROOT.
JIGSAWS_ZIP = f"{DRIVE}/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001.zip"

# Extract to this folder (same directory as the zip; name = zip stem). Created by section 4c.
UNZIP_DIR = f"{DRIVE}/CS5588: DS Capstone/Surgical Detection/JIGSAWS-20260415T004625Z-3-001"

# Filled by section 4c after unzip + auto-detect of JIGSAWS root inside the archive. Leave as-is.
DATA_ROOT = ""

# Alternative: skip the zip — set DATA_ROOT to an existing unzipped JIGSAWS folder and do not run 4c:
# DATA_ROOT = f"{DRIVE}/path/to/folder_with_Suturing_and_Experimental_setup"
# JIGSAWS_ZIP = ""

# SAVE_DIR: where step_*.pt and args.json are written (keep on Drive so runs survive disconnects).
SAVE_DIR = f"{DRIVE}/SS_SD_colab/checkpoints/suturing_expert_lora"

# Code: use exactly one workflow —
#   (A) REPO_URL set → git clone into CLONE_TARGET (/content/SS-SD); PROJECT_ROOT_DRIVE is IGNORED.
#   (B) REPO_URL = "" → use PROJECT_ROOT_DRIVE (you must upload the full SS-SD repo there).
# Default: clone from GitHub so you do not need a Drive copy of the code.
REPO_URL = "https://github.com/ango3636/SS-SD.git"
REPO_BRANCH = "main"
PROJECT_ROOT_DRIVE = f"{DRIVE}/SS-SD"
CLONE_TARGET = "/content/SS-SD"

# Training (defaults mirror typical LoRA runs)
TRAIN_KW = dict(
    expert_only=True,
    train_mode="lora",
    lora_rank=4,
    batch_size=4,
    lr=1e-4,
    epochs=20,
    frame_stride=90,
    image_size=256,
    capture=1,
    save_every=100,
    gradient_accumulation_steps=1,
    num_workers=2,
    model_id="stable-diffusion-v1-5/stable-diffusion-v1-5",
)

# Optional: Hugging Face token for model download (paste in Colab secrets as HF_TOKEN or uncomment)
# import os
# os.environ["HF_TOKEN"] = "hf_..."

## 2. Environment check (GPU)

In [ ]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("No GPU — enable Runtime → Change runtime type → GPU")

## 3. Install dependencies

In [ ]:
# Colab GPU runtimes usually ship torch+CUDA; skip the next line if torch already works.
# %pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu124
%pip install -q diffusers transformers accelerate peft safetensors tqdm scikit-learn opencv-python-headless

## 4. Mount Google Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Not on Colab — skip Drive mount; set DATA_ROOT / SAVE_DIR to local paths.")

## 4b. (Optional) List `MyDrive`

If you are **not** using the zip in section 1, run this after mounting Drive to see folder names and fix paths manually.

In [ ]:
from pathlib import Path

# Run AFTER mounting Drive (section 4).
drive_home = Path("/content/drive/MyDrive")
if not drive_home.is_dir():
    print("Mount Google Drive first (section 4), then re-run this cell.")
else:
    print("Top-level entries under MyDrive (set DATA_ROOT to the JIGSAWS root, not necessarily here):\n")
    for p in sorted(drive_home.iterdir()):
        kind = "dir " if p.is_dir() else "file"
        print(f"  [{kind}] {p.name}")
    print(
        "\nTip: your JIGSAWS root is usually a folder named JIGSAWS or gdrive_cache "
        "containing Suturing, Experimental_setup, etc."
    )

## 4c. Unzip JIGSAWS and set `DATA_ROOT`

Run **after** section 4 (and after section 1 so `JIGSAWS_ZIP` / `UNZIP_DIR` are defined). Extracts **`JIGSAWS_ZIP`** into **`UNZIP_DIR`** (skips re-unzip if that folder already contains a valid JIGSAWS root), then sets **`DATA_ROOT`** for sections 6–7.

If section 1 sets a non-empty **`DATA_ROOT`** to an existing unzipped folder, this cell only validates it and skips the zip.

In [ ]:
import zipfile
from pathlib import Path


def _find_child_dir_ci(parent: Path, name_lower: str) -> Path | None:
    if not parent.is_dir():
        return None
    try:
        for p in parent.iterdir():
            if p.is_dir() and p.name.lower() == name_lower:
                return p
    except OSError:
        return None
    return None


def _discover_jigsaws_root(base: Path) -> Path:
    if _find_child_dir_ci(base, "suturing") and _find_child_dir_ci(base, "experimental_setup"):
        return base.resolve()
    try:
        children = sorted(p for p in base.iterdir() if p.is_dir())
    except OSError as e:
        raise FileNotFoundError(f"Cannot list {base}: {e}") from e
    for child in children:
        if _find_child_dir_ci(child, "suturing") and _find_child_dir_ci(
            child, "experimental_setup"
        ):
            return child.resolve()
    raise FileNotFoundError(
        f"No folder with Suturing + Experimental_setup under {base.resolve()}. "
        "Is this a full JIGSAWS tree?"
    )


manual = (DATA_ROOT or "").strip()
if manual:
    DATA_ROOT = str(_discover_jigsaws_root(Path(manual)))
    print("Using manual DATA_ROOT:", DATA_ROOT)
else:
    zip_path = Path((JIGSAWS_ZIP or "").strip())
    dest = Path(UNZIP_DIR)
    root = None
    if dest.is_dir():
        try:
            root = _discover_jigsaws_root(dest)
        except FileNotFoundError:
            root = None
    if root is None:
        if not zip_path.is_file():
            raise FileNotFoundError(
                f"Zip not found: {zip_path}\n"
                "Upload it to that path, or in section 1 set DATA_ROOT to an existing unzipped JIGSAWS folder."
            )
        dest.mkdir(parents=True, exist_ok=True)
        print("Extracting zip (can take several minutes on Drive)...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(dest)
        root = _discover_jigsaws_root(dest)
    DATA_ROOT = str(root)
    print("DATA_ROOT:", DATA_ROOT)

## 5. Resolve project root (`src/`, `scripts/`)

Run **section 4 (Mount Drive)** first if you use Drive paths. With the default **`REPO_URL`**, the repo is **cloned to** `/content/SS-SD` (nothing required under `MyDrive/SS-SD`). Clear **`REPO_URL`** only if you uploaded the full SS-SD tree to **`PROJECT_ROOT_DRIVE`** on Drive.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path


def resolve_project_root() -> Path:
    url = (REPO_URL or "").strip()
    if url:
        target = Path(CLONE_TARGET)
        marker = target / "scripts" / "train_sd.py"
        if marker.is_file():
            return target.resolve()
        if target.exists() and any(target.iterdir()):
            raise FileNotFoundError(
                f"{target} exists but is not a valid SS-SD checkout (missing scripts/train_sd.py). "
                "Delete that folder or change CLONE_TARGET, then re-run."
            )
        subprocess.run(
            ["git", "clone", "--branch", REPO_BRANCH, url, str(target)],
            check=True,
        )
        if not marker.is_file():
            raise FileNotFoundError(
                f"Git clone finished but {marker} is missing on GitHub. "
                "Push `scripts/train_sd.py` and `src/suturing_pipeline/data/` from your laptop, then re-run, "
                "or set REPO_URL = '' and upload the full SS-SD tree to PROJECT_ROOT_DRIVE on Drive."
            )
        return target.resolve()

    root = Path(PROJECT_ROOT_DRIVE)
    marker_drive = root / "scripts" / "train_sd.py"
    if not marker_drive.is_file():
        hint = (
            f"No scripts/train_sd.py at {root}.\n"
            "You are using REPO_URL = '' (Drive-only code). Either:\n"
            "  1) Set REPO_URL in section 1 to e.g. https://github.com/ango3636/SS-SD.git "
            "(recommended — clones to /content/SS-SD), or\n"
            "  2) Upload the full SS-SD tree to that Drive path so scripts/train_sd.py exists."
        )
        raise FileNotFoundError(hint)
    return root.resolve()


PROJECT_ROOT = resolve_project_root()
os.chdir(PROJECT_ROOT)

_data_pkg = PROJECT_ROOT / "src" / "suturing_pipeline" / "data" / "jigsaws_dataset.py"
if not _data_pkg.is_file():
    raise FileNotFoundError(
        f"Missing {_data_pkg}. Push the full training stack to GitHub or use a Drive copy of the repo."
    )
_help = subprocess.run(
    [sys.executable, str(PROJECT_ROOT / "scripts" / "train_sd.py"), "--help"],
    capture_output=True,
    text=True,
)
if _help.returncode != 0 or "--lora_rank" not in (_help.stdout or ""):
    raise RuntimeError(
        "scripts/train_sd.py --help failed or is outdated.\n"
        f"stdout:\n{_help.stdout}\nstderr:\n{_help.stderr}"
    )
print("PROJECT_ROOT:", PROJECT_ROOT)

## 6. Verify dataset path (`DATA_ROOT`)

Fails fast if `DATA_ROOT` is wrong. Common causes:

- **Shortcut to JIGSAWS** — shortcuts often look empty or missing `suturing/` under Colab. Use a **copied** folder of real files (see intro).
- **One level too shallow** — `DATA_ROOT` must be the folder that **directly** contains the suturing task and experimental-setup trees.
- **Folder name case** — official JIGSAWS zips often use `Suturing/` and `Experimental_setup/`. Linux (Colab) is case-sensitive; this check matches **case-insensitively**, same as `train_sd.py` (`jigsaws_dataset._resolve_ci`).

In [ ]:
from pathlib import Path


def _list_hint(path: Path) -> str:
    if not path.exists():
        return "path does not exist on the mount"
    if not path.is_dir():
        return f"not a directory (symlink/file?): {path}"
    try:
        names = sorted(p.name for p in path.iterdir())
    except OSError as e:
        return f"cannot list directory: {e}"
    head = names[:40]
    extra = f" … (+{len(names) - 40} more)" if len(names) > 40 else ""
    return f"{len(names)} entries: {head}{extra}"


def _find_child_dir_ci(parent: Path, name_lower: str) -> Path | None:
    """Match JIGSAWS layout: e.g. Suturing/ vs suturing/ on case-sensitive Linux."""
    if not parent.is_dir():
        return None
    try:
        for p in parent.iterdir():
            if p.is_dir() and p.name.lower() == name_lower:
                return p
    except OSError:
        return None
    return None


if not (DATA_ROOT or "").strip():
    raise FileNotFoundError(
        "DATA_ROOT is empty. Run section 4c after mounting Drive, or set DATA_ROOT to an unzipped "
        "JIGSAWS folder in section 1."
    )

data = Path(DATA_ROOT)
if not data.is_dir():
    raise FileNotFoundError(
        f"DATA_ROOT is not a directory: {data}\n"
        "Mount Drive (section 4) and run section 4c. If you only added a Drive *shortcut*, copy the real folder into My Drive first.\n"
        f"Debug: {_list_hint(data)}"
    )

required = ("suturing", "experimental_setup")
missing = [key for key in required if _find_child_dir_ci(data, key) is None]
if missing:
    raise FileNotFoundError(
        f"Under DATA_ROOT, expected task folders (any case) for {missing}.\n"
        f"DATA_ROOT={data.resolve()}\n"
        f"Debug listing: {_list_hint(data)}\n\n"
        "If you used a Google Drive *shortcut*: shortcuts often do not mount as real folders in Colab. "
        "Open the dataset in drive.google.com, copy files into a normal My Drive folder, or zip-upload your "
        "local `data/gdrive_cache` and unzip under Drive.\n"
        "If `suturing` appears nested (e.g. DATA_ROOT/JIGSAWS/suturing), set DATA_ROOT one level deeper."
    )

print("DATA_ROOT OK:", data.resolve())
for key in required:
    print(f"  {key!r} ->", _find_child_dir_ci(data, key))

## 7. Run training

Checkpoints: `SAVE_DIR/step_<N>.pt` and `args.json`. Training uses **CUDA** automatically when available (`train_sd.py`).

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.chdir(PROJECT_ROOT)
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    str(PROJECT_ROOT / "scripts" / "train_sd.py"),
    "--data_root",
    str(DATA_ROOT),
    "--save_dir",
    str(SAVE_DIR),
    "--device",
    "cuda",
]
for k, v in TRAIN_KW.items():
    flag = "--" + k  # train_sd.py uses underscores (e.g. --expert_only)
    if isinstance(v, bool):
        if v:
            cmd.append(flag)
    else:
        cmd.extend([flag, str(v)])

print(" ".join(cmd))
_r = subprocess.run(cmd, capture_output=True, text=True)
if _r.stdout:
    print(_r.stdout, end="")
if _r.stderr:
    print(_r.stderr, end="", file=sys.stderr)
if _r.returncode != 0:
    raise RuntimeError(f"train_sd.py exited with code {_r.returncode} (see stderr above).")

## 8. Copy checkpoints back to your laptop / Cursor

1. **Google Drive web UI**: open `SAVE_DIR`, select `step_*.pt` and `args.json`, **Download**.
2. In your SS-SD repo locally, put them under e.g. `checkpoints/suturing_expert_lora/` (create folder if needed).
3. Point scripts at the checkpoint: `--checkpoint checkpoints/suturing_expert_lora/step_1000.pt`.

Optional: zip on Colab for one-shot download:

In [ ]:
import shutil
from pathlib import Path

out_zip = Path(DRIVE) / "SS_SD_colab_ckpt_bundle"
shutil.make_archive(str(out_zip), "zip", root_dir=Path(SAVE_DIR).parent, base_dir=Path(SAVE_DIR).name)
print("Wrote:", str(out_zip) + ".zip")